# How Popular Is Greek Life? — the notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/greek-life-in-america/story_002_greek_life.ipynb)

This notebook reproduces the key statistics in
[How Popular Is Greek Life?](https://pointzerofive.net/2026/02/greek-life-in-america),
story #002 on *pointzerofive*: the applicants-vs-Greek-participation correlation, its
significance test, the public/private split, and the 20-year application surge in a
balanced panel. Run it top to bottom (`Runtime → Run all` in Colab).

In [ ]:
import json
import pathlib
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

BASE = "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/greek-life-in-america/"


def load(name: str):
    for local in (pathlib.Path(name), pathlib.Path("../data") / name):
        if local.exists():  # running inside a checkout
            return json.loads(local.read_text())
    with urllib.request.urlopen(BASE + name) as response:  # Colab / anywhere
        return json.loads(response.read())


scatter = load("us-college-greek-life.json")
uni = pd.DataFrame(scatter["universities"])
print(f"{len(uni)} universities, {uni['applicants'].sum():,} total applicants")
uni.head()

## The cross-section: applicants vs Greek participation

Common Data Set applicant counts (section C1) against each school's reported
fraternity + sorority participation.

In [ ]:
r, p = stats.pearsonr(uni["applicants"], uni["greek_rate"])
n = len(uni)
t = r * np.sqrt(n - 2) / np.sqrt(1 - r**2)
print(f"n = {n}, r = {r:.2f}")
print(f"significance: t = {t:.1f} (df = {n - 2}), p = {p:.4f}")

**But… 24 schools is a small sample. Could r = −0.64 be noise?**

The t-test above answers it: under the null hypothesis (applicant volume unrelated to
Greek participation), a correlation this strong would appear by chance with probability
p ≈ 0.0008. The relationship is statistically real.

**But… is it the same relationship everywhere?** Split by institutional control:

In [ ]:
for control, group in uni.groupby("control"):
    rg, pg = stats.pearsonr(group["applicants"], group["greek_rate"])
    sig = "significant" if pg < 0.05 else "NOT significant at 5%"
    print(f"{control:8s} n = {len(group):2d}  r = {rg:.2f}  p = {pg:.4f}  ({sig})")

In [ ]:
COLORS = {"Public": "#c0392b", "Private": "#3f5b78"}

fig, ax = plt.subplots(figsize=(7, 5))
for control, group in uni.groupby("control"):
    ax.scatter(group["applicants"] / 1000, group["greek_rate"],
               c=COLORS[control], label=control.lower(), s=55, alpha=0.85)
slope, intercept = np.polyfit(uni["applicants"], uni["greek_rate"], 1)
xs = np.linspace(0, uni["applicants"].max(), 50)
ax.plot(xs / 1000, slope * xs + intercept, "--", color="#c0392b", linewidth=1.2, alpha=0.7)
ax.set_xlabel("applicants (thousands, 2022)")
ax.set_ylabel("fraternity + sorority participation (%)")
ax.set_title("Figure 1 — the slope is a public-university story", loc="left", fontsize=11)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

The pattern lives almost entirely in the public flagships (r = −0.85). Among privates the
estimate is imprecise and cannot rule out zero — "not significant" means exactly that,
not "proven absent."

## The 20-year backdrop: a balanced panel

To trace the application surge without fake trend breaks, we keep only schools that
reported admissions in **every** year — a balanced panel.

In [ ]:
longitudinal = load("us-top200-longitudinal.json")
adm = pd.DataFrame(longitudinal["admissions_panel"])
adm = adm[adm["admissions_data_available"] & adm["applicants"].notna() & adm["admitted"].notna()]

years = sorted(adm["year"].unique())
counts = adm.groupby("unitid")["year"].nunique()
balanced_ids = counts[counts == len(years)].index
panel = adm[adm["unitid"].isin(balanced_ids)]

yearly = panel.groupby("year").agg(applicants=("applicants", "sum"), admitted=("admitted", "sum"))
yearly["admit_rate"] = yearly["admitted"] / yearly["applicants"]

growth = yearly["applicants"].iloc[-1] / yearly["applicants"].iloc[0]
print(f"balanced panel: {len(balanced_ids)} schools, {years[0]}-{years[-1]}")
print(f"application growth: {growth:.2f}x")
print(f"admit rate: {yearly['admit_rate'].iloc[0]:.1%} -> {yearly['admit_rate'].iloc[-1]:.1%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(yearly.index, yearly["applicants"] / 1e6, color="#1a1a18", linewidth=2)
ax1.set_ylabel("applications (millions)")
ax1.set_title("Figure 2a — the surge (balanced panel)", loc="left", fontsize=11)
ax2.plot(yearly.index, yearly["admit_rate"] * 100, color="#c0392b", linewidth=2)
ax2.set_ylabel("admit rate (%)")
ax2.set_title("Figure 2b — tightening selectivity", loc="left", fontsize=11)
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---

## Data and methods

- **Scatter:** Common Data Set (section C1) applicant counts and institutional
  fraternity/sorority participation reporting; cross-sectional snapshot.
- **Panel:** IPEDS admissions files, 2001–2022, restricted to schools reporting in every
  year (balanced panel) to keep comparisons like-for-like.
- **Statistics:** Pearson correlations; significance via two-tailed t-test on r
  (`scipy.stats.pearsonr`). Subgroup estimates for small n are imprecise by construction.
- The cross-sectional correlation is **descriptive, not causal**.

*Story: [pointzerofive.net/2026/02/greek-life-in-america](https://pointzerofive.net/2026/02/greek-life-in-america)*